<a href="https://colab.research.google.com/github/mdehghani86/DADS5250-GenAI/blob/main/labs/M06/M06_Lab1_Agent_Concepts_ReAct.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<div style="background: linear-gradient(135deg, #001a70 0%, #0055d4 100%); padding: 30px 36px; border-radius: 14px; margin-bottom: 20px;">
  <h1 style="color: #fff; margin: 0; font-size: 28px;">🤖 M06 Lab 1 — Agent Concepts & ReAct Pattern</h1>
  <p style="color: rgba(255,255,255,0.85); margin: 8px 0 0; font-size: 15px;">DADS 5250: Generative AI in Practice &nbsp;|&nbsp; Prof. Dehghani</p>
  <p style="color: rgba(255,255,255,0.65); margin: 6px 0 0; font-size: 13px;">⭐⭐ Difficulty: Intermediate &nbsp;|&nbsp; ⏱ Time: ~15 min</p>
</div>

<div style="background: #f0f4ff; border-left: 4px solid #0055d4; padding: 16px 20px; border-radius: 0 8px 8px 0; margin: 12px 0;">
  <h3 style="color: #001a70; margin: 0 0 8px;">🎯 Learning Objectives</h3>
  <ol style="margin: 0; color: #1a1a2e; font-size: 14px;">
    <li>Understand the <b>ReAct</b> pattern: Reason → Act → Observe → Repeat</li>
    <li>Build an AI agent <b>from scratch</b> (no framework)</li>
    <li>Define <b>tools</b> that an LLM can invoke (calculator, search, string ops)</li>
    <li>Watch the agent's <b>step-by-step reasoning</b> in real time</li>
    <li>Solve a <b>multi-step problem</b> requiring tool orchestration</li>
  </ol>
</div>

## 📦 Setup

Before we start, let's install the required packages and set up our API connection.

> **🔑 API Key Setup:** Go to the **key icon** (🔑) in the left sidebar of Colab → click **"Add a secret"** → Name: `OPENAI_API_KEY` → Value: your key → Toggle **"Notebook access"** ON.

In [ ]:
# ============================================================
# 📦 Install Dependencies
# ============================================================
!pip install -q --upgrade openai
!pip install -q git+https://github.com/mdehghani86/DADS5250-GenAI.git#subdirectory=utils

In [ ]:
# ============================================================
# 🔑 API Key & Client Setup
# ============================================================
from dads5250 import setup_openai, pp, show_response, show_expected, show_info, quiz

# This reads your OPENAI_API_KEY from Colab Secrets automatically
client = setup_openai()

import json

<div style="background: linear-gradient(135deg, #001a70 0%, #0055d4 100%); padding: 18px 24px; border-radius: 10px; margin: 20px 0 12px;">
  <h2 style="color: white; margin: 0; font-size: 20px;">1️⃣ What Is an AI Agent?</h2>
</div>

So far in this course, we've used LLMs as **stateless responders** — you ask a question, you get an answer. An **agent** is different:

- An agent can **decide** which action to take next
- An agent can **use tools** (calculator, search, databases)
- An agent can **loop** — reason, act, observe, and repeat until the task is done

The most popular pattern for building agents is **ReAct** (Reason + Act):

```
Thought: I need to find the population of Boston.
Action:  web_search("population of Boston 2024")
Observation: Boston has a population of approximately 675,647.
Thought: Now I need to find the number of universities.
Action:  web_search("number of universities in Boston")
Observation: Boston has approximately 35 colleges and universities.
Thought: Now I can divide: 675,647 / 35 = 19,304.
Action:  calculator("675647 / 35")
Observation: 19304.2
Final Answer: The population of Boston divided by the number of universities is approximately 19,304.
```

Today we'll build this **from scratch** — no LangChain, no LangGraph, just raw API calls and Python.

<div style="background: linear-gradient(135deg, #001a70 0%, #0055d4 100%); padding: 18px 24px; border-radius: 10px; margin: 20px 0 12px;">
  <h2 style="color: white; margin: 0; font-size: 20px;">2️⃣ Define Our Tools</h2>
</div>

An agent needs **tools** — Python functions that the LLM can call. Let's define three tools:

1. **Calculator** — evaluates math expressions
2. **Web Search** (mock) — returns simulated search results
3. **String Manipulator** — transforms text (uppercase, reverse, word count)

In [ ]:
# ============================================================
# 🔧 Tool Definitions
# ============================================================

def calculator(expression: str) -> str:
    """Evaluate a math expression safely and return the result."""
    try:
        # Only allow safe math operations
        allowed = set('0123456789+-*/.() ')
        if not all(c in allowed for c in expression):
            return f"Error: Invalid characters in expression"
        result = eval(expression)
        return str(round(result, 4))
    except Exception as e:
        return f"Error: {e}"

def web_search(query: str) -> str:
    """Simulated web search — returns mock results for demo purposes."""
    # Mock knowledge base
    knowledge = {
        "population of boston": "According to the U.S. Census Bureau (2024), Boston has a population of approximately 675,647.",
        "universities in boston": "Boston has approximately 35 colleges and universities, including Harvard, MIT, Boston University, Northeastern, and Boston College.",
        "gdp of usa": "The GDP of the United States in 2024 was approximately $28.78 trillion.",
        "tallest building boston": "The tallest building in Boston is the Prudential Tower at 749 feet (228 m), though One Boston Wharf is under construction at 691 feet.",
        "population of tokyo": "Tokyo has a population of approximately 13.96 million in the city proper, and 37.4 million in the greater metro area.",
        "number of countries": "There are 195 countries in the world — 193 UN member states plus 2 observer states (Vatican City and Palestine).",
    }
    query_lower = query.lower()
    for key, value in knowledge.items():
        if key in query_lower or any(word in query_lower for word in key.split()):
            return value
    return f"No results found for: {query}"

def string_tool(operation: str, text: str) -> str:
    """Manipulate strings: uppercase, lowercase, reverse, word_count, char_count."""
    ops = {
        "uppercase": text.upper(),
        "lowercase": text.lower(),
        "reverse": text[::-1],
        "word_count": str(len(text.split())),
        "char_count": str(len(text)),
    }
    return ops.get(operation, f"Unknown operation: {operation}")

# Quick test
print("🧮 Calculator:", calculator("675647 / 35"))
print("🔍 Search:", web_search("population of boston"))
print("🔤 String:", string_tool("word_count", "Hello world this is a test"))

<div style="background: #fff8e1; border-left: 4px solid #f9a825; padding: 12px 16px; border-radius: 0 8px 8px 0; margin: 12px 0;">
  <b>💡 Pro Tip:</b> In production agents, tools connect to real APIs (Google Search, databases, file systems). Here we use mock data so the lab works without extra API keys — but the <i>pattern</i> is identical.
</div>

<div style="background: linear-gradient(135deg, #001a70 0%, #0055d4 100%); padding: 18px 24px; border-radius: 10px; margin: 20px 0 12px;">
  <h2 style="color: white; margin: 0; font-size: 20px;">3️⃣ The Tool Registry</h2>
</div>

We need to tell the LLM **what tools are available** and **how to call them**. We'll describe each tool in a schema the model understands — this is what OpenAI calls "function calling."

In [ ]:
# ============================================================
# 📋 Tool Registry — Tell the LLM What's Available
# ============================================================

tools_schema = [
    {
        "type": "function",
        "function": {
            "name": "calculator",
            "description": "Evaluate a mathematical expression. Use for any arithmetic, percentages, or numeric computation.",
            "parameters": {
                "type": "object",
                "properties": {
                    "expression": {
                        "type": "string",
                        "description": "Math expression to evaluate, e.g. '675647 / 35' or '(100 * 0.15) + 50'"
                    }
                },
                "required": ["expression"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "web_search",
            "description": "Search the web for factual information. Use when you need to look up data, statistics, or current facts.",
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {
                        "type": "string",
                        "description": "Search query, e.g. 'population of Boston 2024'"
                    }
                },
                "required": ["query"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "string_tool",
            "description": "Manipulate text strings. Operations: uppercase, lowercase, reverse, word_count, char_count.",
            "parameters": {
                "type": "object",
                "properties": {
                    "operation": {
                        "type": "string",
                        "enum": ["uppercase", "lowercase", "reverse", "word_count", "char_count"],
                        "description": "The string operation to perform"
                    },
                    "text": {
                        "type": "string",
                        "description": "The text to operate on"
                    }
                },
                "required": ["operation", "text"]
            }
        }
    }
]

# Map tool names to functions for easy dispatch
tool_functions = {
    "calculator": lambda args: calculator(args["expression"]),
    "web_search": lambda args: web_search(args["query"]),
    "string_tool": lambda args: string_tool(args["operation"], args["text"]),
}

pp({"tools": [t["function"]["name"] for t in tools_schema], "count": len(tools_schema)}, "Registered Tools")

<div style="background: linear-gradient(135deg, #001a70 0%, #0055d4 100%); padding: 18px 24px; border-radius: 10px; margin: 20px 0 12px;">
  <h2 style="color: white; margin: 0; font-size: 20px;">4️⃣ The ReAct Agent Loop</h2>
</div>

This is the core of the agent. The loop works like this:

1. Send the user's question + tool schemas to the LLM
2. If the LLM returns a **tool call** → execute the tool → feed result back → repeat
3. If the LLM returns a **text response** → we're done (final answer)

The LLM is the "brain" that decides what to do at each step. Our Python code is the "body" that executes the actions.

In [ ]:
# ============================================================
# 🔄 The ReAct Agent Loop
# ============================================================

def react_agent(user_question: str, max_steps: int = 10, verbose: bool = True) -> str:
    """
    A ReAct agent that reasons, acts, and observes in a loop.
    Returns the final answer as a string.
    """
    # Initialize conversation with a system prompt
    messages = [
        {
            "role": "system",
            "content": (
                "You are a helpful assistant with access to tools. "
                "Use tools to look up facts and perform calculations. "
                "Always verify your answers with tools rather than guessing. "
                "Think step by step."
            )
        },
        {"role": "user", "content": user_question}
    ]

    if verbose:
        print(f"\n{'='*70}")
        print(f"🧑 User: {user_question}")
        print(f"{'='*70}")

    for step in range(1, max_steps + 1):
        # Call the LLM with tool schemas
        response = client.chat.completions.create(
            model="gpt-4.1-mini",
            messages=messages,
            tools=tools_schema,
            temperature=0
        )

        choice = response.choices[0]
        message = choice.message

        # Case 1: LLM wants to call tool(s)
        if message.tool_calls:
            # Add the assistant's message (with tool calls) to history
            messages.append(message)

            for tool_call in message.tool_calls:
                fn_name = tool_call.function.name
                fn_args = json.loads(tool_call.function.arguments)

                if verbose:
                    print(f"\n🔵 Step {step} — Reason & Act")
                    print(f"   🛠️  Tool: {fn_name}")
                    print(f"   📥 Args: {fn_args}")

                # Execute the tool
                if fn_name in tool_functions:
                    result = tool_functions[fn_name](fn_args)
                else:
                    result = f"Error: Unknown tool '{fn_name}'"

                if verbose:
                    print(f"   👁️  Observation: {result}")

                # Feed the tool result back to the LLM
                messages.append({
                    "role": "tool",
                    "tool_call_id": tool_call.id,
                    "content": result
                })

        # Case 2: LLM returns a text response (final answer)
        else:
            final = message.content
            if verbose:
                print(f"\n🟢 Final Answer (after {step} step{'s' if step > 1 else ''})")
                print(f"   🤖 {final}")
                print(f"{'='*70}")
            return final

    return "Agent exceeded maximum steps."

print("✅ ReAct agent ready!")

<div style="background: linear-gradient(135deg, #001a70 0%, #0055d4 100%); padding: 18px 24px; border-radius: 10px; margin: 20px 0 12px;">
  <h2 style="color: white; margin: 0; font-size: 20px;">5️⃣ Agent in Action — Simple Questions</h2>
</div>

Let's start with some simple questions that require one tool call each.

In [ ]:
# ============================================================
# 🧪 Test 1: Simple Math
# ============================================================
react_agent("What is 1234 * 5678?")

In [ ]:
# ============================================================
# 🧪 Test 2: Simple Search
# ============================================================
react_agent("What is the population of Tokyo?")

In [ ]:
# ============================================================
# 🧪 Test 3: String Operation
# ============================================================
react_agent("How many words are in the sentence: 'The quick brown fox jumps over the lazy dog'?")

<div style="background: linear-gradient(135deg, #001a70 0%, #0055d4 100%); padding: 18px 24px; border-radius: 10px; margin: 20px 0 12px;">
  <h2 style="color: white; margin: 0; font-size: 20px;">6️⃣ Multi-Step Problem</h2>
</div>

Now the real test — a question that requires **multiple tools** in sequence. Watch how the agent reasons through each step.

In [ ]:
# ============================================================
# 🧪 Multi-Step: Search + Search + Calculate
# ============================================================
react_agent(
    "What is the population of Boston divided by the number of universities in Boston?"
)

In [ ]:
# ============================================================
# 🧪 Multi-Step: Search + Calculate + String
# ============================================================
react_agent(
    "Look up the GDP of the USA, divide it by the number of countries in the world, "
    "and tell me the result. Also, how many characters are in the word 'Northeastern'?"
)

**📝 Your Observations:** *(double-click to edit)*

1. Watch the reasoning steps above. Did the agent always pick the right tool first? When did it backtrack or chain tools? _[Your answer]_
2. How many steps did the multi-step problem take? Was the final answer correct? _[Your answer]_
3. What happens if you ask a question that none of the tools can answer (e.g., "Tell me a joke")? _[Your answer]_

<div style="background: #f0f4ff; border-left: 4px solid #0055d4; padding: 14px 18px; border-radius: 0 8px 8px 0; margin: 12px 0; font-size: 14px;">
  <b>🧠 Key Takeaway:</b> The LLM <b>decides</b> what tool to use and in what order — we never hard-coded the sequence. The agent autonomously breaks down the problem, calls the right tools, and synthesizes the results into a final answer. This is the foundation of all AI agents.
</div>

<div style="background: linear-gradient(135deg, #001a70 0%, #0055d4 100%); padding: 18px 24px; border-radius: 10px; margin: 20px 0 12px;">
  <h2 style="color: white; margin: 0; font-size: 20px;">📝 Knowledge Check</h2>
</div>

In [ ]:
# ============================================================
# 📝 Quiz — Test Your Understanding
# ============================================================

quiz([
    {
        "q": "What is the correct order of steps in the ReAct pattern?",
        "options": [
            "Act → Observe → Reason → Repeat",
            "Reason → Act → Observe → Repeat",
            "Observe → Reason → Act → Repeat",
            "Plan → Execute → Verify → Report"
        ],
        "answer": 1,
        "explanation": "ReAct stands for Reason + Act. The agent first REASONS about what to do, then ACTS by calling a tool, then OBSERVES the result, and REPEATS until it has a final answer."
    },
    {
        "q": "In our agent, what determines whether the loop continues or stops?",
        "options": [
            "A fixed number of iterations (always runs exactly 3 times)",
            "The user sends a 'stop' command",
            "If the LLM returns tool_calls, we continue; if it returns text, we stop",
            "The agent stops when all tools have been used at least once"
        ],
        "answer": 2,
        "explanation": "The agent loop checks the LLM's response: if it contains tool_calls, we execute them and continue. If it contains a plain text response (no tool calls), the agent is done and returns the final answer."
    }
])

<div style="background: linear-gradient(135deg, #001a70 0%, #0055d4 100%); padding: 18px 24px; border-radius: 10px; margin: 20px 0 12px;">
  <h2 style="color: white; margin: 0; font-size: 20px;">🔧 Hands-On Exercise</h2>
</div>

### Exercise: Add a Date Calculator Tool

Add a new tool called `date_calculator` that computes the number of days between two dates or tells you what day of the week a given date falls on.

Complete the code below. Replace each `-----` with the correct value.

In [ ]:
# --- Expected Output ---
show_expected("date_calculator('day_of_week', '2024-12-25') → Wednesday\ndate_calculator('days_between', '2024-01-01', '2024-12-31') → 365 days")

In [ ]:
# ============================================================
# 🔧 Exercise: Add a Date Calculator Tool
# ============================================================
# Replace each ----- with the correct value

from datetime import datetime, timedelta

def date_calculator(operation: str, date1: str, date2: str = "") -> str:
    """Perform date calculations. Operations: days_between, day_of_week, add_days."""
    try:
        d1 = datetime.strptime(date1, "%Y-%m-%d")
        if operation == "days_between" and date2:
            d2 = datetime.strptime(date2, "%Y-%m-%d")
            return str(abs((d2 - d1).days)) + " days"
        elif operation == "day_of_week":
            return d1.strftime("%A")
        elif operation == "add_days" and date2:
            result = d1 + timedelta(days=int(date2))
            return result.strftime("%Y-%m-%d")
        else:
            return "Invalid operation or missing parameters"
    except Exception as e:
        return f"Error: {e}"

# 1. Add the schema for the new tool
date_tool_schema = {
    "type": "-----",                    # What type of tool is this?
    "function": {
        "name": "date_calculator",
        "description": "Perform date calculations: days between dates, day of the week, add days.",
        "parameters": {
            "type": "object",
            "properties": {
                "operation": {
                    "type": "string",
                    "enum": ["days_between", "day_of_week", "-----"],  # What's the 3rd operation?
                    "description": "The date operation to perform"
                },
                "date1": {"type": "string", "description": "First date in YYYY-MM-DD format"},
                "date2": {"type": "string", "description": "Second date or number of days to add"}
            },
            "required": ["-----", "date1"]  # Which fields are required?
        }
    }
}

# 2. Add to the registry
extended_tools = tools_schema + [date_tool_schema]
extended_functions = {**tool_functions}
extended_functions["-----"] = lambda args: date_calculator(  # What's the tool name?
    args["operation"], args["date1"], args.get("date2", "")
)

print(f"✅ Now have {len(extended_tools)} tools: {[t['function']['name'] for t in extended_tools]}")

# 3. Test it!
print("\n📅 Quick test:", date_calculator("day_of_week", "2024-12-25"))
print("📅 Quick test:", date_calculator("days_between", "2024-01-01", "2024-12-31"))

**📝 Your Observations:** *(double-click to edit this cell)*

1. How does adding a new tool change the agent's capabilities? _[Your answer]_

2. What would happen if two tools had overlapping functionality? How would the agent decide? _[Your answer]_

3. What's the downside of giving an agent too many tools? _[Your answer]_

<div style="background: #e8f5e9; border-left: 4px solid #4caf50; padding: 14px 18px; border-radius: 0 8px 8px 0; margin: 12px 0;">
  <h3 style="margin: 0 0 8px;">📖 Self-Study Activity</h3>
  <p style="margin: 0; font-size: 14px;">Try these extensions on your own:</p>
  <ol style="font-size: 14px;">
    <li><b>Conversation agent:</b> Modify the agent to remember previous questions (add chat history)</li>
    <li><b>Error handling:</b> What happens when a tool returns an error? Make the agent retry or try a different approach</li>
    <li><b>New tools:</b> Add a weather lookup tool and a unit converter — then ask a multi-tool question</li>
  </ol>
</div>

<div style="background: linear-gradient(135deg, #001a70 0%, #0055d4 100%); padding: 24px 30px; border-radius: 12px; margin: 24px 0 0; text-align: center;">
  <h2 style="color: white; margin: 0 0 10px;">🎉 Lab 1 Complete!</h2>
  <p style="color: rgba(255,255,255,0.85); margin: 0; font-size: 14px;">You built a ReAct agent from scratch — tools, reasoning loop, and multi-step problem solving.</p>
  <div style="background: rgba(255,255,255,0.15); border-radius: 8px; padding: 12px; margin-top: 14px; text-align: left;">
    <p style="color: white; margin: 0 0 6px; font-weight: bold; font-size: 14px;">Key Takeaways:</p>
    <ul style="color: rgba(255,255,255,0.9); margin: 0; font-size: 13px;">
      <li><b>ReAct</b> = Reason → Act → Observe → Repeat</li>
      <li>Tools are Python functions with <b>JSON schemas</b> the LLM can understand</li>
      <li>The LLM <b>decides</b> which tool to call — we don't hard-code the logic</li>
      <li>The agent loop continues until the LLM returns a <b>final text answer</b></li>
    </ul>
  </div>
  <p style="color: rgba(255,255,255,0.7); margin: 14px 0 0; font-size: 13px;"><b>Next →</b> M06 Lab 2: LangGraph Agents</p>
</div>